<a href="https://colab.research.google.com/github/brianhphillips/testrepo/blob/main/IMDb_Stunt_Coordinator_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

# Step 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Adjust to your IMDb dataset folder path in Drive
base_path = "/content/drive/MyDrive/"

Mounted at /content/drive


In [2]:
# Step 2 — Target stunt coordinators
target_names = {
    "Sam Hargrave",
    "George Marshall Ruge",
    "Damon Caro"
}

In [3]:
# Step 3 — Get nconst IDs for these names using chunking
name_basics_file = base_path + "name.basics.tsv"
name_ids = {}

chunksize = 100000
for chunk in pd.read_csv(name_basics_file, sep='\t', dtype=str, na_values='\\N', chunksize=chunksize):
    filtered = chunk[chunk['primaryName'].isin(target_names)]
    for _, row in filtered.iterrows():
        name_ids[row['nconst']] = row['primaryName']

print("Found nconst IDs:", name_ids)

Found nconst IDs: {'nm0138902': 'Damon Caro', 'nm0741396': 'George Marshall Ruge', 'nm1092087': 'Sam Hargrave', 'nm13660968': 'Sam Hargrave', 'nm14571303': 'Sam Hargrave'}


In [4]:
# Step 4 — Filter principals for Stunt Coordinator roles
principals_file = base_path + "title.principals.tsv"
stunt_titles = []

for chunk in pd.read_csv(principals_file, sep='\t', dtype=str, na_values='\\N', chunksize=chunksize):
    # Keep rows where nconst is in our list and category contains 'stunt coordinator'
    mask = chunk['nconst'].isin(name_ids.keys()) & chunk['category'].str.lower().str.contains('stunt coordinator', na=False)
    filtered = chunk[mask]
    stunt_titles.extend(list(filtered[['nconst', 'tconst']].itertuples(index=False, name=None)))

print(f"Total matching credits found: {len(stunt_titles)}")

Total matching credits found: 0


In [5]:
# Step 5 — Convert to DataFrame
stunt_df = pd.DataFrame(stunt_titles, columns=['nconst', 'tconst'])

In [6]:
# Step 6 — Load ratings and merge
ratings_file = base_path + "title.ratings.tsv"
ratings_chunks = pd.read_csv(ratings_file, sep='\t', dtype={'tconst': str, 'averageRating': float}, na_values='\\N', chunksize=chunksize)

# We'll collect ratings for only the stunt_df's tconst
tconst_set = set(stunt_df['tconst'])
ratings_list = []

for chunk in ratings_chunks:
    filtered = chunk[chunk['tconst'].isin(tconst_set)]
    ratings_list.append(filtered)

ratings_df = pd.concat(ratings_list, ignore_index=True)

In [7]:
# Step 7 — Merge stunt_df with ratings
merged = stunt_df.merge(ratings_df, on='tconst', how='left')

In [8]:
# Step 8 — Map names back
merged['primaryName'] = merged['nconst'].map(name_ids)

In [9]:
# Step 9 — Group by person and calculate average rating
avg_ratings = merged.groupby('primaryName')['averageRating'].mean().reset_index()
avg_ratings = avg_ratings.rename(columns={'averageRating': 'avgIMDbScore'}).sort_values('avgIMDbScore', ascending=False)

print(avg_ratings)

Empty DataFrame
Columns: [primaryName, avgIMDbScore]
Index: []
